<a href="https://colab.research.google.com/github/tacinunesc/Aula_1_Gemeos_Digitais_UniFacens/blob/main/aula3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**1. Instalando dependencias necessárias**

In [1]:
!pip install transformers torch scikit-learn -q

**2. Importando Bibliotecas**

In [2]:
# Importando bibliotecas principais
from transformers import AutoTokenizer, AutoModel
import torch
import torch.nn.functional as F
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

**3. Definindo o texto**

In [3]:
# Texto que será analisado
text = "O gato subiu no telhado"

**4. Tokenização**

In [4]:
# Carregando o tokenizer do BERT
# BERT é um modelo baseado em Transformers treinado para entender linguagem natural
# "base" indica tamanho médio (~110M parâmetros)
# "uncased" significa que ignora maiúsculas/minúsculas
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

# Convertendo o texto em tokens (palavras ou partes de palavras)
tokens = tokenizer.tokenize(text)

# Convertendo tokens em IDs numéricos
token_ids = tokenizer.convert_tokens_to_ids(tokens)

# Exibindo resultados
print("Texto original:", text)
print("Tokens:", tokens)
print("Token IDs:", token_ids)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Texto original: O gato subiu no telhado
Tokens: ['o', 'ga', '##to', 'sub', '##iu', 'no', 'tel', '##had', '##o']
Token IDs: [1051, 11721, 3406, 4942, 17922, 2053, 10093, 16102, 2080]


**5. Converter para entrada do modelo**

In [5]:
# Convertendo texto para formato que o modelo entende
inputs = tokenizer(text, return_tensors="pt")

# Exibindo estrutura de entrada
print(inputs)

{'input_ids': tensor([[  101,  1051, 11721,  3406,  4942, 17922,  2053, 10093, 16102,  2080,
           102]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])}


**O que é esse dicionário de saída?**
- Isso {
 'input_ids': tensor([[...]]),
 'token_type_ids': tensor([[...]]),
 'attention_mask': tensor([[...]])
} é a representação completa da frase para o modelo.
- O modelo não recebe texto — ele recebe três matrizes que dizem o que é cada token, de onde ele vem e o que deve ser considerado.
- Isso são os tokens convertidos para números: input_ids': tensor([[101, 1051, 11721, ... ,102]])
- Observou que são diferentes do início? Foram incluídos tokens especiais: 101 → [CLS]
102 → [SEP], eles marcarm o início e o fim da frase.
- 'token_type_ids': tensor([[0, 0, 0, 0, ...]]) indica a qual frase o token pertence, pois poderíamos ter duas frases.
- 'attention_mask': tensor([[1, 1, 1, 1, ...]]) demonstra quais tokens são válidos, podemos ter frases de tamanhos diferentes e uma matriz precisa ser um retângulo perfeito. Se a Frase A tem 3 palavras e a Frase B tem 5, a "tabela" fica torta. O PAD preenche os espaços vazios para que todas as linhas tenham o mesmo comprimento. Nesse caso tudo é 1 pois só temos uma frase.
- Eficiência da GPU: As GPUs realizam cálculos paralelos. Quando você entrega um bloco uniforme (ex: 32 frases de 128 tokens cada), ela consegue processar tudo de uma vez. Sem o padding, você teria que enviar uma frase por vez, o que deixaria o processo muito lento.
- Exemplo: frase 1: "O gato"            -> [O, gato, PAD, PAD] -> Mask: [1, 1, 0, 0], Frase 2: "O gato no telhado" -> [O, gato, no, telhado] -> Mask: [1, 1, 1, 1]

**6. Carregar modelo BERT**

In [6]:
# Carregando modelo BERT pré-treinado
model = AutoModel.from_pretrained("bert-base-uncased")

# Passando o texto pelo modelo
outputs = model(**inputs)

# Pegando os embeddings (representação vetorial)
embeddings = outputs.last_hidden_state

# Exibindo dimensões
print("Shape dos embeddings:", embeddings.shape)

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Shape dos embeddings: torch.Size([1, 11, 768])


O "bert-base-uncased" é uma versão específica do BERT (Bidirectional Encoder Representations from Transformers), um modelo de inteligência artificial desenvolvido pelo Google para processar e entender a linguagem humana.
Aqui está o que cada parte do nome significa no contexto de programação com Python:

    - BERT: É a arquitetura baseada em Transformers. Ele é "bidirecional" porque analisa o contexto de uma palavra observando tanto o que vem antes quanto o que vem depois dela em uma frase.
    - base: Refere-se ao tamanho do modelo. A versão "base" é menor e mais leve, contendo 12 camadas (blocos de transformadores) e cerca de 110 milhões de parâmetros.
    - large: 24 camadas, 340 milhões de parametros
    - uncased: Significa que o modelo não diferencia maiúsculas de minúsculas. Antes de processar o texto, ele transforma tudo em minúsculas e remove acentos. Por exemplo, "Brasil" e "brasil" são tratados como a mesma palavra.

Observação: O AutoModel.from_pretrained, está carregando apenas o "corpo" do BERT (os pesos que entendem a linguagem) para gerar os embeddings. As "cabeças" de classificação (as camadas cls.predictions e cls.seq_relationship) não são necessárias para isso, por isso o Python diz que elas são UNEXPECTED (inesperadas).

**7. Interpretar os embeddings**

In [7]:
# Cada token agora é representado por um vetor de 768 dimensões
print("Número de tokens:", embeddings.shape[1])
print("Dimensão de cada vetor:", embeddings.shape[2])

Número de tokens: 11
Dimensão de cada vetor: 768


**8. Similiaridade entre tokens**

In [8]:
# Convertendo embeddings para numpy
emb = embeddings[0].detach().numpy()

# Calculando similaridade entre tokens
similarity_matrix = cosine_similarity(emb)

# Exibindo matriz
print("Matriz de similaridade entre tokens:")
print(similarity_matrix)

Matriz de similaridade entre tokens:
[[ 1.0000001   0.34413782  0.13820738  0.17327799  0.15996578  0.20347235
   0.17479981  0.2199733   0.1222958   0.1731197  -0.0036287 ]
 [ 0.34413782  0.9999997   0.5924298   0.71488464  0.60412616  0.69579256
   0.76028574  0.57976013  0.55967575  0.5926276   0.11801811]
 [ 0.13820738  0.5924298   0.9999999   0.58504915  0.64204645  0.6173426
   0.6090683   0.5894519   0.5767511   0.44236314  0.07447856]
 [ 0.17327799  0.71488464  0.58504915  1.          0.6379912   0.8136123
   0.86582464  0.61746025  0.59802026  0.7328117   0.11959408]
 [ 0.15996578  0.60412616  0.64204645  0.6379912   0.99999976  0.6941167
   0.6774025   0.6998488   0.63440776  0.51693606  0.07590821]
 [ 0.20347235  0.69579256  0.6173426   0.8136123   0.6941167   1.0000001
   0.8047304   0.66272604  0.6729391   0.70633763  0.10441145]
 [ 0.17479981  0.76028574  0.6090683   0.86582464  0.6774025   0.8047304
   1.          0.6295913   0.6261476   0.7515053   0.14485975]
 [ 0.2199

- É essa matriz que as LLMs utilizam para entender a relação entre as pálavras, produto das operações algébricas com as matrizes Q e K. Esse é o mapa semantico da nossa frase!

**9. Selecionar o último token**

In [9]:
# Selecionando o vetor do último token
# Este vetor representa todo o contexto da frase
last_token_vector = embeddings[0][-1]

print("Vetor do último token:")
print(last_token_vector[:10])  # mostrando apenas os primeiros valores

Vetor do último token:
tensor([ 0.9165, -0.1706, -0.1538,  0.5235, -0.6905, -0.4572,  0.6215, -0.6523,
         0.8087, -0.3673], grad_fn=<SliceBackward0>)


- Lembrando que o último token é quem carregaria todo o contexto da frase inteira, é o token já transformado, que já passou pelas matrizes Q, K e V.

**10. Simular W_vocab**

In [10]:
# Criando uma matriz aleatória simulando o vocabulário
# Dimensão:
# 768 -> tamanho do vetor
# 1000 -> número de palavras simuladas
W_vocab = torch.randn(768, 1000)

- Nessa matrriz que criamos, cada coluna representaria uma palavra, então teríamos 1000 palavras aqui.

**11. Calcular logits**

In [11]:
# Multiplicação matricial
# Estamos comparando o vetor do contexto com todas as palavras
logits = last_token_vector @ W_vocab

print("Shape dos logits:", logits.shape)

Shape dos logits: torch.Size([1000])


- Agora vem o próximo segredo da LLM, vamos ver quais palavras combinam com todo contexto e informação carregada na nossa frase incial.

**12. Converter para probabilidade (Softmax)**

In [12]:
# Convertendo logits em probabilidades
probs = F.softmax(logits, dim=0)

# Convertendo para numpy para visualização
probs_np = probs.detach().numpy()

# Mostrando algumas probabilidades
print("Primeiras probabilidades:")
print(probs_np[:10])

Primeiras probabilidades:
[6.8343829e-12 7.4389669e-17 7.7639709e-26 6.7429130e-17 1.7179594e-16
 6.4094267e-21 2.2191451e-13 3.1062573e-01 3.2086592e-27 7.6708606e-32]


- Aqui já o vetor resultante, um vetor de scores chamado logits. Isso representa scores não normalizados que indicam o quanto cada palavra combina com o contexto. É como um ranking bruto:
  - telhado → 12.3
  - sofá → 2.1
  - avião → -3.7

**13. Ver as palavras mais prováveis**

In [13]:
# Pegando os índices das maiores probabilidades
top_k = 5
top_indices = np.argsort(probs_np)[-top_k:][::-1]

print("Top palavras simuladas (estas são de um vocabulário aleatório e não se referem ao GPT-2):")
for i, idx in enumerate(top_indices):
    print(f"Rank {i+1} - índice {idx} - probabilidade {probs_np[idx]:.4f}")

# Para sua pergunta sobre o GPT-2 gerar 'yu' em vez de 'telhado':
# O modelo 'gpt2' é primariamente treinado em inglês, o que explica a dificuldade em gerar palavras em português.
# Para obter 'telhado' como saída, você precisaria de um modelo de linguagem treinado em português.

Top palavras simuladas (estas são de um vocabulário aleatório e não se referem ao GPT-2):
Rank 1 - índice 349 - probabilidade 0.6507
Rank 2 - índice 7 - probabilidade 0.3106
Rank 3 - índice 685 - probabilidade 0.0326
Rank 4 - índice 389 - probabilidade 0.0026
Rank 5 - índice 348 - probabilidade 0.0012


- Agora, aqui já temos o vetor com as probabilidades de cada palavra, só precisamos definir qual queremos adotar.
- Nós utilizamos uma matriz aleatória W-vocab, não é real. Então não há palavras associadas de verdade.
- Simulando as palvras, ficaria algo assim (só para ilustrar):

In [14]:
vocab_simulado = [f"palavra_{i}" for i in range(1000)]

for i, idx in enumerate(top_indices):
    print(vocab_simulado[idx], probs_np[idx])

palavra_349 0.6507174
palavra_7 0.31062573
palavra_685 0.032621153
palavra_389 0.0025721316
palavra_348 0.001245929


**14. Conclusão**

Resumo do processo:

1. Texto → tokens
2. Tokens → números
3. Números → vetores (embeddings)
4. Vetores → relações (similaridade)
5. Último vetor representa o contexto
6. Contexto é comparado com todas as palavras (W_vocab)
7. Resultado → logits
8. Softmax → probabilidades
9. Escolha da próxima palavra

Conhecemos é o funcionamento básico de uma LLM.

**Mas e se eu quiser fazer para valer?**

1. Instalar dependências

In [15]:
!pip install transformers torch -q

2. Importar bibliotecas

In [16]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
import torch.nn.functional as F

3. Carregar modelo e tokenizer

In [17]:
# Carregando tokenizer (converte texto ↔ números)
tokenizer = AutoTokenizer.from_pretrained("gpt2")

# Carregando modelo GPT2 (modelo gerador de texto)
model = AutoModelForCausalLM.from_pretrained("gpt2")

# Colocando em modo avaliação (desliga treino)
model.eval()

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)

4. Definir texto de entrada

In [18]:
text = "O gato subiu no"

# Converter para formato do modelo
inputs = tokenizer(text, return_tensors="pt")

print(inputs)

{'input_ids': tensor([[   46,   308,  5549,   850, 16115,   645]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1]])}


- Note que mudaram os valores dos tokens!

5. Rodar o modelo

In [19]:
# Passando o texto pelo modelo
with torch.no_grad():
    outputs = model(**inputs)

6. Entender a saída

In [20]:
# Pegando os logits (scores para cada palavra)
logits = outputs.logits

print("Shape dos logits:", logits.shape)

Shape dos logits: torch.Size([1, 6, 50257])


- Alguma diferença para o exemplo anterior?
- 1 (Batch Size): Enviamos 1 frase para o modelo.
- 6 (Sequence Length): A frase de entrada foi dividida em 6 tokens. Para cada um desses tokens, o GPT-2 calculou uma previsão.
- 50257 (Vocab Size): Este é o "tamanho do dicionário" do GPT-2. Ele conhece 50.257 palavras (ou pedaços de palavras) e está relacionando nossos tokens a todas elas.


A grande diferença técnica em relação ao exemplo anterior (BERT) está no terceiro número do Shape:

    No BERT (768): O foco era a dimensão do conhecimento (Embedding). O modelo estava tentando "entender" o significado profundo daquelas palavras.
    No GPT-2 (50257): O foco é a previsão do vocabulário. O modelo está literalmente dando uma nota (score) para cada uma das 50.257 palavras que ele conhece, tentando adivinhar qual delas se encaixa melhor naquela posição.

**Por que 50257 e não 768?**
- O GPT-2 usa uma camada extra no final (chamada de Language Modeling Head). Ela pega os 768 de "conhecimento" e projeta isso contra todo o dicionário dele para gerar os logits.
- Dica: Se você rodar tokenizer.decode([50256]), verá o token especial <|endoftext|>, que é o último item desse dicionário gigante!
Quer que eu te mostre como o GPT-2 escolhe de fato a palavra (usando algo chamado top_k ou temperature) em vez de só pegar a mais provável?

**O que o modelo está te dizendo com isso?**

Para cada um dos seus 6 tokens, o modelo gerou uma lista de 50257 números.

    Quanto maior o número (score) em uma posição, mais o GPT-2 acha que aquela palavra do dicionário deveria vir a seguir.
    Lembrando que Logits são valores "crus" (podem ser negativos ou muito altos). Para transformá-los em probabilidades (0% a 100%), costumamos usar uma função Softmax.

7. Pegar o último token (próxima palavra)

In [21]:
# Pegando os logits do último token
last_token_logits = logits[0, -1, :]

print(last_token_logits.shape)

torch.Size([50257])


- Esse vetor contém um score para cada palavra do vocabulário do GPT2.

8. Converter para probabilidades

In [22]:
probs = F.softmax(last_token_logits, dim=0)

9. Top palavras mais prováveis

In [23]:
# Pegar top 10 palavras
top_k = 10
top_probs, top_indices = torch.topk(probs, top_k)

# Converter IDs para palavras
for i in range(top_k):
    token_id = top_indices[i].item()
    token = tokenizer.decode([token_id])
    prob = top_probs[i].item()

    print(f"{i+1}: '{token}' → {prob:.4f}")

1: ' k' → 0.0570
2: ' o' → 0.0256
3: ' t' → 0.0227
4: ' sh' → 0.0203
5: ' n' → 0.0185
6: ' no' → 0.0175
7: ' j' → 0.0171
8: ' h' → 0.0159
9: ' wa' → 0.0155
10: ' ch' → 0.0131


10. Veja a mágica que está revolucionando nosso mundo acontecer:

In [24]:
# Gerar continuação
output = model.generate(**inputs, max_length=10)

# Converter para texto
generated_text = tokenizer.decode(output[0])

print("Texto gerado:")
print(generated_text)

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Texto gerado:
O gato subiu no kara no k


**- O que está acontecendo???** 😧
- Isso é literalmente o modelo dizendo:

- “isso aqui continua sendo provável… então vou repetir”

- kara → virou contexto → gerou kara → virou contexto → gerou kara

Como rodamos "output = model.generate(**inputs, max_length=20)", no Default:
- do_sample=False: ou seja → modo determinístico (greedy)
- O modelo fez: pega a palavra MAIS provável → SEMPRE
- Então: caiu em um ciclo de alta probabilidade

**o modelo está maximizando localmente:**
- não globalmente
- ele não “planeja frase”
- ele “anda passo a passo”

11. Rodar com sampling

In [25]:
output = model.generate(
    **inputs,
    max_length=20,
    do_sample=True,
    top_k=50,
    top_p=0.9,
    temperature=0.8
)

# Converter para texto
generated_text = tokenizer.decode(output[0])

print("Texto gerado:")
print(generated_text)

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Texto gerado:
O gato subiu no oi nga nga nga nga nga nga


- Olha que interessante, mais problemas. O que está acontecendo agora?
- Imagina o motivo?

**O que aconteceu aqui!?**
- O que as LLMs fazem?
- Geram a continuação mais provável e coerente para a frase que inserimos.
- O modelo pierreguillou/gpt2-small-portuguese foi treinado em uma vasta quantidade de texto em português e, para ele, "rio", "mas não foi visto" e "estação de trem" são sequências perfeitamente válidas e naturais depois de "O gato subiu no". Ele não tenta adivinhar uma palavra única, mas sim construir uma frase completa e gramaticalmente correta.

**Veja que a frase parou abruptamente**
- O parâmetro max_length=20 que usamos na função generate instrui o modelo a gerar até 20 tokens (palavras ou pedaços de palavras) após a sua entrada.
- Isso resulta em uma frase mais longa, e não apenas uma única palavra.
- O que acontece se alterarmos esse parametro?



12. Vamos rodar com um modelo que tem vobabulário em português

### 12.1 Carregar o modelo e tokenizer em português

In [26]:
# Carregando tokenizer e modelo GPT-2 em português
tokenizer_pt = AutoTokenizer.from_pretrained("pierreguillou/gpt2-small-portuguese")
model_pt = AutoModelForCausalLM.from_pretrained("pierreguillou/gpt2-small-portuguese")

# Colocando em modo avaliação
model_pt.eval()

print("Modelo GPT-2 em português e tokenizer carregados!")

config.json:   0%|          | 0.00/666 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/92.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/120 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/510M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/510M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/149 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie transformer.wte.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
GPT2LMHeadModel LOAD REPORT from: pierreguillou/gpt2-small-portuguese
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...11}.attn.masked_bias | UNEXPECTED |  | 
transformer.h.{0...11}.attn.bias        | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Modelo GPT-2 em português e tokenizer carregados!


In [28]:
text_pt = "O gato subiu no"

# Converter para formato do modelo em português
inputs_pt = tokenizer_pt(text_pt, return_tensors="pt")

# Gerar continuação com sampling
output_pt = model_pt.generate(
    **inputs_pt,
    max_length=50,
    do_sample=True,
    top_k=50,
    top_p=0.9,
    temperature=0.7,
    pad_token_id=tokenizer_pt.eos_token_id # Definir pad_token_id para o tokenizer em português
)

# Converter para texto
generated_text_pt = tokenizer_pt.decode(output_pt[0], skip_special_tokens=True)

print("Texto gerado em português:")
print(generated_text_pt)

Texto gerado em português:
O gato subiu no telhado, mas ele não era um gato, ele era uma cobra. Depois de lutar com o gato, ele foi atacado por um gato, e então ele foi jogado fora da sala. No entanto, ele foi salvo por um gato


**- Que mórbido!**
- Note que ele está continuando a frase, na direção que acha mais provável.

**- Agora veja o que os argumentos controlam e veja o que acontece com as respostas**

    - do_sample=True (ou False): Este é o controle fundamental. Se False (o padrão se você não especificar), o modelo opera em modo greedy ou determinístico, ou seja, em cada passo ele simplesmente escolhe o token com a maior probabilidade. Isso muitas vezes leva a repetições (como vimos com o 'kara') ou a textos previsíveis. Se True, o modelo começa a amostrar (sample) palavras das distribuições de probabilidade, introduzindo aleatoriedade e tornando as gerações mais variadas e criativas. É o primeiro passo para sair do modo 'repetitivo'.
    
    - temperature (temperatura): Pense nisso como a 'temperatura' da criatividade do modelo. Ela ajusta a distribuição de probabilidade dos tokens:
        Temperatura baixa (ex: 0.2 - 0.5): Torna a distribuição mais 'afiada', ou seja, os tokens mais prováveis ficam ainda mais prováveis e os menos prováveis, muito menos prováveis. Isso resulta em textos mais conservadores, previsíveis e focados nas opções de alta probabilidade.
        Temperatura alta (ex: 0.8 - 1.0+): 'Suaviza' a distribuição, tornando as probabilidades mais uniformes. Isso aumenta a chance de tokens menos prováveis serem selecionados, resultando em textos mais diversos, criativos e, às vezes, menos coerentes.

    - top_k: Limita o número de tokens que o modelo pode escolher. Em cada passo, o modelo só considera os k tokens com as maiores probabilidades para a amostragem. Por exemplo, se top_k=50, ele só irá considerar os 50 tokens mais prováveis, ignorando todos os outros, mesmo que a probabilidade do 51º token não seja tão baixa. Isso ajuda a evitar a geração de palavras completamente aleatórias e sem sentido.

    - top_p (nucleus sampling): Este é um método de amostragem mais dinâmico que top_k. Em vez de um número fixo de tokens, top_p seleciona o menor conjunto de tokens cuja soma das probabilidades é maior ou igual a p. Por exemplo, se top_p=0.9, o modelo seleciona os tokens mais prováveis até que a soma de suas probabilidades atinja 90%. Isso permite que o tamanho do conjunto de tokens válidos varie em cada etapa de geração, adaptando-se melhor ao contexto e evitando tanto a previsibilidade excessiva quanto a aleatoriedade extrema.

Como eles funcionam juntos?

- Quando do_sample=True, você geralmente usa temperature em conjunto com top_k ou top_p (ou ambos).

- top_k e top_p atuam como filtros para o conjunto de tokens que podem ser amostrados, enquanto a temperature 'modela' a distribuição de probabilidade dentro desse conjunto filtrado.

- Juntos, eles oferecem um controle granular sobre o estilo e a qualidade da geração de texto do LLM.

### 13.1 Poderíamos ir para o Ollama, mas temos limitações no Colab

- **Recursos:** Modelos grandes via Ollama podem exigir muita VRAM e RAM, que podem ser limitadas ou voláteis no ambiente do Colab.
- **Configuração:** Ollama é projetado para rodar localmente e configurar um servidor Ollama dentro de um contêiner do Colab para então baixar e servir modelos pode ser complexo e levar tempo.
- **Alternativa:** A biblioteca `transformers` já otimiza o uso direto de modelos pré-treinados, aproveitando a GPU do Colab de forma mais eficiente sem a necessidade de uma camada adicional como Ollama para esta finalidade.

### 13.2 Carregar e usar um modelo GPT-2 maior (inglês)

Vamos agora carregar uma versão maior do GPT-2. Usaremos o `gpt2-large` para mostrar a diferença na qualidade da geração. Como este modelo é treinado principalmente em inglês, usaremos um prompt em inglês para aproveitar ao máximo suas capacidades.

In [29]:
# Carregando tokenizer e modelo GPT-2 Large
tokenizer_large = AutoTokenizer.from_pretrained("gpt2-large")
model_large = AutoModelForCausalLM.from_pretrained("gpt2-large")

# Colocando em modo avaliação
model_large.eval()

print("Modelo GPT-2 Large e tokenizer carregados!")

config.json:   0%|          | 0.00/666 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/3.25G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/436 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2-large
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...35}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Modelo GPT-2 Large e tokenizer carregados!


In [30]:
text_en = "The cat climbed onto the"

# Converter para formato do modelo
inputs_en = tokenizer_large(text_en, return_tensors="pt")

# Gerar continuação com sampling
output_en = model_large.generate(
    **inputs_en,
    max_length=50,
    do_sample=True,
    top_k=50,
    top_p=0.9,
    temperature=0.7,
    pad_token_id=tokenizer_large.eos_token_id # Definir pad_token_id para evitar warnings
)

# Converter para texto
generated_text_en = tokenizer_large.decode(output_en[0], skip_special_tokens=True)

print("Texto gerado com GPT-2 Large:")
print(generated_text_en)

Texto gerado com GPT-2 Large:
The cat climbed onto the hood of the car, and then leapt from the hood to the ground.

The cat continued to climb onto the hood of the car and then jumped onto the hood of the car.

The cat jumped onto the


Note que o `gpt2-large` gera texto muito mais coerente e natural em inglês, o que demonstra a capacidade de modelos com mais parâmetros. Para obter resultados igualmente bons em português, precisariamos de um modelo grande treinado especificamente nessa língua, o que exigiria ainda mais recursos.